In [ ]:
"""Day 21 / Session 3 - Lab Assignment 2: Bidirectional LSTM Wrapper from Scratch."""
import numpy as np


def sigmoid(x):
    x = np.asarray(x, dtype=float)
    e = np.exp(-np.abs(x))
    return np.where(x >= 0, 1.0 / (1.0 + e), e / (1.0 + e))


def lstm_cell_step(x_t, h_prev, c_prev, Wx, Wh, b):
    H = len(h_prev)
    z = np.asarray(Wx) @ np.asarray(x_t) + np.asarray(Wh) @ np.asarray(h_prev) + np.asarray(b)
    i = sigmoid(z[0:H])
    f = sigmoid(z[H:2 * H])
    g = np.tanh(z[2 * H:3 * H])
    o = sigmoid(z[3 * H:4 * H])
    c_t = f * np.asarray(c_prev) + i * g
    h_t = o * np.tanh(c_t)
    return h_t, c_t


def lstm_run(X, Wx, Wh, b):
    H = np.asarray(Wh).shape[1]
    h = np.zeros(H)
    c = np.zeros(H)
    states = []
    for x_t in np.asarray(X, dtype=float):
        h, c = lstm_cell_step(x_t, h, c, Wx, Wh, b)
        states.append(h)
    return np.array(states)                              # (T, H) in input order


def bilstm_forward(X, Wxf, Whf, bf, Wxb, Whb, bb):
    X = np.asarray(X, dtype=float)
    Hf = lstm_run(X, Wxf, Whf, bf)                       # reads left to right
    Hb = lstm_run(X[::-1], Wxb, Whb, bb)[::-1]           # reads right to left, then re-aligned to input order
    return np.concatenate([Hf, Hb], axis=1)              # (T, 2H)
